In [ ]:
# colab_02_global_model.py  (v4 - seed toplulugu)
# Rossmann - 1115 magaza tek model, 28 gun ileri, p10/p50/p90
# 3 seed egitilir, quantile'lar ortalanir (varyans azaltma)

import os, json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ------------------------------------------------------------------
# 1. AYARLAR
# ------------------------------------------------------------------
DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
HAZIRLIK = f"{DIZIN}/hazirlik"
CIKTI = f"{DIZIN}/model"
os.makedirs(CIKTI, exist_ok=True)

EMB_BOYUT = 24
GRU_BIRIM = 128
COZUCU_BIRIM = 128
DROPOUT = 0.3
L2 = 1e-5

BATCH = 256
EPOCH = 200
LR = 3e-4
PATIENCE = 20

QUANTILES = [0.10, 0.50, 0.90]
SEEDLER = [42, 1337, 2024]

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TF: 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ------------------------------------------------------------------
# 2. VERI
# ------------------------------------------------------------------
meta = json.load(open(f"{HAZIRLIK}/meta.json"))
GECMIS, UFUK = meta["gecmis"], meta["ufuk"]
N_MAGAZA = meta["n_magaza"]
magaza_ort = np.array(meta["magaza_ort"], dtype=np.float32)
magaza_std = np.array(meta["magaza_std"], dtype=np.float32)
bosluklu = set(meta.get("bosluklu_magazalar", []))

def yukle(ad):
    d = np.load(f"{HAZIRLIK}/nn_{ad}.npz")
    return {k: d[k] for k in d.files}

egitim, val, test = yukle("egitim"), yukle("val"), yukle("test")

N_GECMIS_KANAL = egitim["X_gecmis"].shape[-1]
N_GELECEK_KANAL = egitim["X_gelecek"].shape[-1]

assert N_GECMIS_KANAL == len(meta["gecmis_kanal"]), "Gecmis kanal uyusmazligi!"
assert N_GELECEK_KANAL == len(meta["gelecek_kanal"]), "Gelecek kanal uyusmazligi!"

print(f"egitim {egitim['X_gecmis'].shape} | val {val['X_gecmis'].shape} | test {test['X_gecmis'].shape}")
print(f"gelecek kanal: {N_GELECEK_KANAL} -> {meta['gelecek_kanal']}")
print(f"BEKLENEN adim/epoch: {-(-len(egitim['y']) // BATCH)}")

def hedef_paketle(d):
    return np.stack([d["y"], d["maske"]], axis=-1).astype(np.float32)

def girdi_paketle(d):
    return {"gecmis": d["X_gecmis"], "gelecek": d["X_gelecek"], "magaza": d["X_magaza"]}

egitim (282472, 56, 5) | val (5575, 56, 5) | test (1115, 56, 5)
gelecek kanal: 14 -> ['acik', 'promo', 'okul', 'tatil', 'promo2', 'dow_sin', 'dow_cos', 'ay_sin', 'ay_cos', 'dom_sin', 'dom_cos', 'h_norm', 'gecen_yil_olcek', 'gecen_yil_acik']
BEKLENEN adim/epoch: 1104


In [ ]:
# ------------------------------------------------------------------
# 3. MASKELI PINBALL KAYIP
# ------------------------------------------------------------------
def pinball_maskeli(y_paket, y_pred):
    y = y_paket[..., 0:1]
    m = y_paket[..., 1:2]
    q = tf.constant(QUANTILES, dtype=tf.float32)
    hata = y - y_pred
    kayip = tf.maximum(q * hata, (q - 1.0) * hata) * m
    return tf.reduce_sum(kayip) / (tf.reduce_sum(m) * len(QUANTILES) + 1e-6)

def mae_medyan(y_paket, y_pred):
    """Izleme metrigi - OLCEKLI uzayda. Rapor metrikleri bolum 6'da, ham EUR uzerinde."""
    y, m, p50 = y_paket[..., 0], y_paket[..., 1], y_pred[..., 1]
    return tf.reduce_sum(tf.abs(y - p50) * m) / (tf.reduce_sum(m) + 1e-6)

In [ ]:
# ------------------------------------------------------------------
# 4. MODEL
# ------------------------------------------------------------------
def model_kur():
    g_in = keras.Input(shape=(GECMIS, N_GECMIS_KANAL), name="gecmis")
    f_in = keras.Input(shape=(UFUK, N_GELECEK_KANAL), name="gelecek")
    s_in = keras.Input(shape=(), dtype="int32", name="magaza")

    emb = layers.Embedding(N_MAGAZA, EMB_BOYUT,
                           embeddings_regularizer=keras.regularizers.l2(L2),
                           name="magaza_embedding")(s_in)
    emb = layers.Flatten(name="magaza_vektor")(emb)

    h = layers.GRU(GRU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(g_in)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.GRU(GRU_BIRIM, kernel_regularizer=keras.regularizers.l2(L2))(h)
    h = layers.Dropout(DROPOUT)(h)

    baglam = layers.Concatenate()([h, emb])
    baglam = layers.Dense(COZUCU_BIRIM, activation="relu")(baglam)
    baglam_seq = layers.RepeatVector(UFUK)(baglam)

    d = layers.Concatenate()([f_in, baglam_seq])
    d = layers.GRU(COZUCU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(d)
    d = layers.Dropout(DROPOUT)(d)
    d = layers.TimeDistributed(layers.Dense(64, activation="relu"))(d)
    cikis = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name="quantiles")(d)

    return keras.Model([g_in, f_in, s_in], cikis, name="rossmann_global")

model_kur().summary()

Model: "rossmann_global"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gecmis (InputLayer) │ (None, 56, 5)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 56, 128)   │     51,840 │ gecmis[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 56, 128)   │          0 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ magaza (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128)       │     99,072 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ magaza_embedding    │ (None, 24)        │     26,760 │ magaza[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ magaza_vektor       │ (None, 24)        │          0 │ magaza_embedding… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 152)       │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ magaza_vektor[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     19,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gelecek             │ (None, 28, 14)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 28, 128)   │          0 │ dense[0][0]       │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 28, 142)   │          0 │ gelecek[0][0],    │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 28, 128)   │    104,448 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 28, 128)   │          0 │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 28, 64)    │      8,256 │ dropout_2[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantiles           │ (None, 28, 3)     │        195 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 310,155 (1.18 MB)

 Trainable params: 310,155 (1.18 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# ------------------------------------------------------------------
# 5. EGITIM - SEED TOPLULUGU
# ------------------------------------------------------------------
def egit_tek(seed):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    m = model_kur()
    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss=pinball_maskeli, metrics=[mae_medyan])
    cb = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                      restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=8, min_lr=1e-5, verbose=0),
    ]
    h = m.fit(girdi_paketle(egitim), hedef_paketle(egitim),
              validation_data=(girdi_paketle(val), hedef_paketle(val)),
              epochs=EPOCH, batch_size=BATCH, shuffle=True, callbacks=cb, verbose=2)
    en_iyi = int(np.argmin(h.history["val_loss"]) + 1)
    print(f"  -> seed {seed}: en iyi epoch {en_iyi}/{len(h.history['loss'])}, "
          f"val_loss {min(h.history['val_loss']):.4f}")
    return m, h

modeller, gecmisler = [], []
t0 = time.time()
for sd in SEEDLER:
    print(f"\n{'='*50}\nSEED {sd}\n{'='*50}")
    m, h = egit_tek(sd)
    m.save_weights(f"{CIKTI}/global_seed{sd}.weights.h5")
    modeller.append(m)
    gecmisler.append(h)
sure_dk = (time.time() - t0) / 60
print(f"\nToplam egitim: {sure_dk:.1f} dk ({len(SEEDLER)} model)")

def tahmin_al(d, secim=None):
    """secim=None -> topluluk (quantile ortalamasi) | secim=i -> tek model"""
    ms = modeller if secim is None else [modeller[secim]]
    return np.mean([m.predict(girdi_paketle(d), batch_size=512, verbose=0)
                    for m in ms], axis=0)


SEED 42
Epoch 1/200
1104/1104 - 34s - 31ms/step - loss: 0.1455 - mae_medyan: 0.4370 - val_loss: 0.1031 - val_mae_medyan: 0.3145 - learning_rate: 3.0000e-04
Epoch 2/200
1104/1104 - 26s - 23ms/step - loss: 0.1078 - mae_medyan: 0.3263 - val_loss: 0.0984 - val_mae_medyan: 0.3001 - learning_rate: 3.0000e-04
Epoch 3/200
1104/1104 - 40s - 36ms/step - loss: 0.0997 - mae_medyan: 0.3020 - val_loss: 0.0984 - val_mae_medyan: 0.2995 - learning_rate: 3.0000e-04
Epoch 4/200
1104/1104 - 41s - 37ms/step - loss: 0.0955 - mae_medyan: 0.2895 - val_loss: 0.0985 - val_mae_medyan: 0.2993 - learning_rate: 3.0000e-04
Epoch 5/200
1104/1104 - 25s - 23ms/step - loss: 0.0928 - mae_medyan: 0.2814 - val_loss: 0.1000 - val_mae_medyan: 0.3035 - learning_rate: 3.0000e-04
Epoch 6/200
1104/1104 - 25s - 23ms/step - loss: 0.0909 - mae_medyan: 0.2759 - val_loss: 0.1008 - val_mae_medyan: 0.3053 - learning_rate: 3.0000e-04
Epoch 7/200
1104/1104 - 25s - 22ms/step - loss: 0.0894 - mae_medyan: 0.2714 - val_loss: 0.1026 - val_ma

In [ ]:
# ------------------------------------------------------------------
# 6. TERS OLCEKLEME + DEGERLENDIRME (ham EUR uzerinde)
# ------------------------------------------------------------------
def ters_olcek(y_olcek, magaza_idx):
    ort = magaza_ort[magaza_idx][:, None]
    std = magaza_std[magaza_idx][:, None]
    return np.expm1(y_olcek * std + ort)

def smape(gercek, tahmin, maske):
    g, t = gercek[maske], tahmin[maske]
    if g.size == 0:
        return float("nan")
    payda = (np.abs(g) + np.abs(t)) / 2.0
    ok = payda > 1e-6
    return 100.0 * np.mean(np.abs(g[ok] - t[ok]) / payda[ok])

def wmape(gercek, tahmin, maske):
    g, t = gercek[maske], tahmin[maske]
    if g.size == 0:
        return float("nan")
    return 100.0 * np.abs(g - t).sum() / max(np.abs(g).sum(), 1e-6)

def yanlilik(gercek, tahmin, maske):
    g, t = gercek[maske], tahmin[maske]
    if g.size == 0:
        return float("nan")
    return 100.0 * (t.sum() - g.sum()) / max(g.sum(), 1e-6)

def kapsama(gercek, alt, ust, maske):
    return 100.0 * np.mean((gercek[maske] >= alt[maske]) & (gercek[maske] <= ust[maske]))

def degerlendir(d, ad, secim=None, magaza_kirilimi=False, sessiz=False):
    p = tahmin_al(d, secim)
    maske = d["maske"].astype(bool)
    gercek = d["y_ham"]
    p10 = ters_olcek(p[..., 0], d["X_magaza"])
    p50 = ters_olcek(p[..., 1], d["X_magaza"])
    p90 = ters_olcek(p[..., 2], d["X_magaza"])

    s = smape(gercek, p50, maske)
    w = wmape(gercek, p50, maske)
    yan = yanlilik(gercek, p50, maske)
    kap = kapsama(gercek, p10, p90, maske)
    ufuk_smape = [smape(gercek[:, h:h+1], p50[:, h:h+1], maske[:, h:h+1]) for h in range(UFUK)]
    ufuk_wmape = [wmape(gercek[:, h:h+1], p50[:, h:h+1], maske[:, h:h+1]) for h in range(UFUK)]

    if sessiz:
        print(f"  {ad:<22} sMAPE {s:.2f}% | WMAPE {w:.2f}% | kapsama {kap:.1f}%")
    else:
        print(f"\n=== {ad} ===")
        print(f"sMAPE (p50)          : {s:.2f}%   (gun bazli esit agirlik)")
        print(f"WMAPE (p50)          : {w:.2f}%   (ciro agirlikli - is metrigi)")
        print(f"Yanlilik             : {yan:+.2f}%  (+ fazla / - eksik tahmin)")
        print(f"p10-p90 kapsama      : {kap:.1f}%   (hedef 80%)")
        print(f"1. gun  sMAPE/WMAPE  : {ufuk_smape[0]:.2f}% / {ufuk_wmape[0]:.2f}%")
        print(f"28. gun sMAPE/WMAPE  : {ufuk_smape[-1]:.2f}% / {ufuk_wmape[-1]:.2f}%")
        print(f"1-7 gun  ort         : {np.mean(ufuk_smape[:7]):.2f}% / {np.mean(ufuk_wmape[:7]):.2f}%")
        print(f"22-28 gun ort        : {np.mean(ufuk_smape[21:]):.2f}% / {np.mean(ufuk_wmape[21:]):.2f}%")

    cikti = dict(smape=float(s), wmape=float(w), yanlilik=float(yan), kapsama=float(kap),
                 ufuk_smape=[float(x) for x in ufuk_smape],
                 ufuk_wmape=[float(x) for x in ufuk_wmape])

    if magaza_kirilimi:
        no = d["magaza_no"]
        satirlar = []
        for m_no in np.unique(no):
            sec = (no == m_no)
            mk = maske[sec]
            if mk.sum() == 0:
                continue
            satirlar.append((int(m_no),
                             smape(gercek[sec], p50[sec], mk),
                             wmape(gercek[sec], p50[sec], mk),
                             kapsama(gercek[sec], p10[sec], p90[sec], mk)))
        arr = np.array(satirlar, dtype=float)
        m_smape = arr[:, 1]
        print(f"\n--- magaza bazli sMAPE dagilimi ({len(arr)} magaza) ---")
        for q in [5, 25, 50, 75, 95]:
            print(f"  p{q:<3}: {np.percentile(m_smape, q):.2f}%")
        en_kotu = arr[np.argsort(-m_smape)][:10]
        print("  en kotu 10:", ", ".join(f"{int(r[0])}({r[1]:.1f}%)" for r in en_kotu))
        if bosluklu:
            bm = np.isin(arr[:, 0].astype(int), list(bosluklu))
            print(f"  veri boslugu olan {int(bm.sum())} magaza : {m_smape[bm].mean():.2f}%")
            print(f"  temiz gecmisli {int((~bm).sum())} magaza : {m_smape[~bm].mean():.2f}%")
            cikti["bosluklu_smape"] = float(m_smape[bm].mean())
            cikti["temiz_smape"] = float(m_smape[~bm].mean())
        cikti["magaza_smape"] = {int(r[0]): float(r[1]) for r in arr}
        cikti["magaza_wmape"] = {int(r[0]): float(r[2]) for r in arr}
        cikti["magaza_kapsama"] = {int(r[0]): float(r[3]) for r in arr}
    return cikti

sonuc = {}

# --- tekil seed'ler: kosu gurultusunu OLC ---
print(f"\n{'='*50}\nTEKIL SEED SONUCLARI (gurultu bandi)\n{'='*50}")
for i, sd in enumerate(SEEDLER):
    sonuc[f"val_seed_{sd}"] = degerlendir(val, f"VAL  seed {sd}", secim=i, sessiz=True)
    sonuc[f"test_seed_{sd}"] = degerlendir(test, f"TEST seed {sd}", secim=i, sessiz=True)

t_tekil = [sonuc[f"test_seed_{s}"]["smape"] for s in SEEDLER]
print(f"\nTekil test sMAPE : {[f'{x:.2f}' for x in t_tekil]}")
print(f"Yayilim          : {max(t_tekil)-min(t_tekil):.2f} puan  <-- gurultu bandi")
print(f"Ortalama         : {np.mean(t_tekil):.2f}%")

# --- topluluk ---
sonuc["val"] = degerlendir(val, "VALIDATION (topluluk)", magaza_kirilimi=True)
sonuc["test"] = degerlendir(test, f"TEST (topluluk) {meta['test_hedef'][0]}..{meta['test_hedef'][1]}",
                            magaza_kirilimi=True)

print(f"\n{'='*50}")
print(f"Topluluk kazanci : {np.mean(t_tekil) - sonuc['test']['smape']:+.2f} puan "
      f"(tekil ort {np.mean(t_tekil):.2f}% -> topluluk {sonuc['test']['smape']:.2f}%)")
print(f"{'='*50}")

In [ ]:
# ------------------------------------------------------------------
# 7. KAYIT
# ------------------------------------------------------------------
emb_ort = np.mean([m.get_layer("magaza_embedding").get_weights()[0] for m in modeller], axis=0)
np.save(f"{CIKTI}/magaza_embedding.npy",
        modeller[0].get_layer("magaza_embedding").get_weights()[0])  # gorsellestirme: seed 42
print(f"\nembedding matrisi: {emb_ort.shape}  (gorsel icin seed {SEEDLER[0]} kaydedildi)")

sonuc["seedler"] = SEEDLER
sonuc["tekil_test_smape"] = [float(x) for x in t_tekil]
sonuc["gurultu_bandi"] = float(max(t_tekil) - min(t_tekil))
sonuc["egitim_gecmisi"] = {
    f"seed_{sd}": {k: [float(x) for x in v] for k, v in h.history.items()}
    for sd, h in zip(SEEDLER, gecmisler)}
sonuc["en_iyi_epochlar"] = [int(np.argmin(h.history["val_loss"]) + 1) for h in gecmisler]
sonuc["parametre_sayisi"] = int(modeller[0].count_params())
sonuc["egitim_suresi_dk"] = float(sure_dk)
sonuc["mimari"] = dict(emb=EMB_BOYUT, gru=GRU_BIRIM, cozucu=COZUCU_BIRIM,
                       dropout=DROPOUT, l2=L2, batch=BATCH, lr=LR,
                       patience=PATIENCE, seedler=SEEDLER)
sonuc["kanallar"] = {"gecmis": meta["gecmis_kanal"], "gelecek": meta["gelecek_kanal"]}

with open(f"{CIKTI}/global_model_sonuc.json", "w") as f:
    json.dump(sonuc, f, indent=2)

print("\nKaydedildi ->", CIKTI)
print("Agirliklar:", [f"global_seed{s}.weights.h5" for s in SEEDLER])